# TRIBE v2 → short-video engagement (Colab)

Runs [neuro-engagement-forecasting](https://github.com/RehanKrish21/Neuro-engagement-forecasts) on a Colab GPU: Meta's TRIBE v2 predicts brain responses to each video, and we test whether those predictions forecast real engagement on [SnapUGC](https://github.com/dasongli1/SnapUGC_Engagement) better than simple baselines.

**Before you start**
1. *Runtime → Change runtime type →* **A100** or **L4** GPU. The free T4 may run out of memory.
2. Get access to [meta-llama/Llama-3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B). Create a **Read** token at huggingface.co/settings/tokens and add it as a Colab secret named **`HF_TOKEN`** (🔑 icon, left sidebar; switch on notebook access).

**How to run:** top to bottom. Step 1 restarts Python once; when it tells you to, just run it again.

No Google Drive access is needed. Everything stays on the Colab machine, and step 7 downloads your results. Files are lost if the runtime disconnects, so for long runs use the [Modal script](https://github.com/RehanKrish21/Neuro-engagement-forecasts/blob/main/cloud/modal_app.py).

*TRIBE v2 is CC BY-NC 4.0 (non-commercial). Check SnapUGC's terms too.*

## 1. Setup
Run this cell. The **first time**, it installs everything (a few minutes) and then restarts Python. Colab may show *"Your session crashed"*, which is expected. **Run it again** until it prints ✅ Ready.

In [ ]:
import importlib.metadata as md, os, subprocess, sys

REPO = "https://github.com/RehanKrish21/Neuro-engagement-forecasts.git"
PKGS = [
    "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git",
    f"neuroengage @ git+{REPO}",
    "pyarrow", "gdown",
]
FORCE_REINSTALL = False   # set True to pull the latest version of this repo

def version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

def run(cmd):
    """Run a shell command, streaming its output into the notebook."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("GPU:", gpu or "NONE: switch the runtime to a GPU before continuing")

installed = (version("tribev2") and version("neuroengage")
             and (version("torch") or "").startswith("2.6"))
if installed and not FORCE_REINSTALL:
    import torch, numpy, tribev2, neuroengage
    from neuroengage.tribe_runner import TribeExtractor  # noqa: F401
    print(f"✅ Ready: torch {torch.__version__}, numpy {numpy.__version__}, "
          f"CUDA {torch.cuda.is_available()}, neuroengage from {os.path.dirname(neuroengage.__file__)}")
else:
    extra = ["--reinstall-package", "neuroengage"] if FORCE_REINSTALL else []
    code = run(["uv", "pip", "install", "--system", *extra, *PKGS])
    if code != 0:
        print("\nuv failed, retrying with pip ...\n")
        code = run([sys.executable, "-m", "pip", "install", *(["--force-reinstall", "--no-deps"] if FORCE_REINSTALL else []), *PKGS])
    if code != 0:
        raise RuntimeError("Install failed: copy the error lines above and send them over.")
    print("\nInstalled. Restarting Python now. RUN THIS CELL AGAIN after the restart.")
    sys.stdout.flush()
    os.kill(os.getpid(), 9)

## 2. Settings and Hugging Face login
Also checks your Llama 3.2 access up front, so a permissions problem shows here and not 20 minutes into a run.

In [ ]:
from pathlib import Path
import logging

SPLIT = "train"      # SnapUGC split: "train" or "val"
N = 300              # videos to run (start small, check timing, then raise)
SEED = 0

WORK = Path("/content/runs")           # outputs (deliberately NOT named "neuroengage")
DATA = Path("/content/snapugc_data")   # downloaded + unpacked SnapUGC
CACHE = Path("/content/tribe_cache")   # TRIBE's per-video feature cache
for p in (WORK, DATA, CACHE):
    p.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)
for noisy in ("httpx", "urllib3", "huggingface_hub", "filelock"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

from huggingface_hub import login, hf_hub_download
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception as exc:
    print(f"No usable HF_TOKEN secret ({type(exc).__name__}); paste your token instead:")
    login()

try:
    hf_hub_download("meta-llama/Llama-3.2-3B", "config.json")
    print("✅ Hugging Face login OK, Llama 3.2 access OK")
except Exception as exc:
    raise RuntimeError(
        "Can't download Llama 3.2. Either access isn't approved yet "
        "(huggingface.co/meta-llama/Llama-3.2-3B) or the token is fine-grained without "
        "'read access to public gated repos'. Use a Read token."
    ) from exc

## 3. Smoke test: one video
Loads TRIBE v2 once (the first time it downloads several GB of models, ~10–20 min) and runs it on a 52-second trailer. The `extractor` object is reused in step 5, so the models aren't loaded twice.

In [ ]:
import time, urllib.request
import pandas as pd
from neuroengage.tribe_runner import TribeExtractor

sample = Path("/content/sample.mp4")
if not sample.exists():
    urllib.request.urlretrieve("https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4", sample)

t0 = time.time()
extractor = TribeExtractor(cache_folder=CACHE)
print(f"Models loaded in {time.time() - t0:.0f} s")

t0 = time.time()
smoke = extractor.run(pd.DataFrame({"video_id": ["sintel"], "path": [str(sample)]}),
                      WORK / "smoke", overwrite=True)
secs = time.time() - t0
print(f"✅ TRIBE v2 works: {len(smoke)} predicted seconds in {secs:.0f} s "
      "(first run includes one-time model downloads; real timing is measured in step 5)")
smoke.head()

## 4. Download SnapUGC
Pulls the split from the Google Drive links in the [SnapUGC repo](https://github.com/dasongli1/SnapUGC_Engagement/blob/main/dataset/readme.md) and unpacks it, then finds the label file and video folder automatically.

If `gdown` fails with a quota error, download the file in your browser, upload it into `/content/snapugc_data/download/` (folder icon on the left), and re-run.

In [ ]:
import gdown, glob, shutil
from collections import Counter

FILE_IDS = {"train": "1Mv5Esq5gGuxRTayabRUb5NmHwEN3JdbD", "val": "1iY7qUfqivYzUqUID501dKQDpI3NQil7y"}
dl_dir = DATA / "download"
unpacked = DATA / SPLIT
dl_dir.mkdir(exist_ok=True)

if not any(unpacked.rglob("*.mp4")):
    archives = [p for p in dl_dir.iterdir() if p.is_file()]
    if archives:
        archive = archives[0]
    else:  # trailing "/" keeps the original file name, so the format can be detected
        archive = Path(gdown.download(id=FILE_IDS[SPLIT], output=f"{dl_dir}/", quiet=False))
    print(f"Unpacking {archive.name} ({archive.stat().st_size / 1e9:.1f} GB) ...")
    shutil.unpack_archive(str(archive), str(unpacked))

csvs = sorted(glob.glob(f"{unpacked}/**/*.csv", recursive=True))
video_dirs = Counter(os.path.dirname(p) for p in glob.glob(f"{unpacked}/**/*.mp4", recursive=True))
print("Label files:", csvs)
print("Video folders:", [(d, n) for d, n in video_dirs.most_common(3)])
LABELS = csvs[0]                          # change if a different CSV holds the labels
VIDEOS = video_dirs.most_common(1)[0][0]
pd.read_csv(LABELS).head()

## 5. Run TRIBE v2 on a stratified subset
Picks `N` videos spread across the full engagement range, then runs TRIBE on each. Progress is logged per video. **Re-running resumes**: finished videos are skipped. Videos that fail are skipped and listed in `failures.csv`.

In [ ]:
from neuroengage.snapugc import build_manifest, load_labels

RUN = WORK / f"snapugc_{SPLIT}_n{N}_seed{SEED}"
RUN.mkdir(parents=True, exist_ok=True)
manifest_path = RUN / "manifest.csv"
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path, dtype={"video_id": str})
else:
    manifest = build_manifest(load_labels(LABELS), VIDEOS, n=N, seed=SEED)
    manifest.to_csv(manifest_path, index=False)
print(f"{len(manifest)} videos in this run")

# Time the first 5 videos (models are warm now) to estimate the full run.
t0 = time.time()
pilot = extractor.run(manifest.head(5), RUN / "work", table_name="network_timeseries_pilot")
per_video = (time.time() - t0) / max(pilot["video_id"].nunique(), 1)
print(f"⏱  ~{per_video:.0f} s per video → ~{per_video * len(manifest) / 3600:.1f} h for all {len(manifest)}")

t0 = time.time()
timeseries = extractor.run(manifest, RUN / "work")   # the 5 pilot videos are reused from cache
print(f"✅ {timeseries['video_id'].nunique()}/{len(manifest)} videos processed in {(time.time() - t0) / 60:.1f} min")


## 6. Brain features vs baselines
For each target (NAWP = how much people watch, ECR = share who stay past 5 s): `baseline` (length + title/description stats) vs `brain` (TRIBE features) vs `baseline+brain`, scored by Spearman correlation on held-out videos.

In [ ]:
from IPython.display import Image, Markdown, display
from neuroengage.pipeline import run_score_training

ts_videos = set(timeseries["video_id"])
res = run_score_training(
    manifest[manifest["video_id"].isin(ts_videos)], timeseries, ["nawp", "ecr"], RUN / "results",
    baseline_cols=["duration_s", "title_n_words", "has_description"],
    hook_seconds=5, seed=SEED,
    title=f"SnapUGC ({len(ts_videos)} videos): TRIBE v2 brain features vs baselines",
)
display(Image(str(RUN / "results" / "summary.png")))
display(Markdown((RUN / "results" / "report.md").read_text()))

**Reading it:** the brain features help only if `baseline+brain` beats `baseline` by more than `spearman_std` in most folds (`folds_better_than_baseline` near 1.0). With a few hundred videos the error bars are wide, so treat a first run as a pilot and scale `N` up before concluding anything.

## 7. Download results
Zips the report, figure, metrics and per-video TRIBE predictions. Do this before closing the tab.

In [ ]:
from google.colab import files
zip_path = shutil.make_archive(f"/content/{RUN.name}", "zip", RUN)
print(f"{Path(zip_path).stat().st_size / 1e6:.0f} MB")
files.download(zip_path)